In [1]:

!pip install pandas==2.2.2 bitsandbytes accelerate transformers -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 67.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [34]:
import pandas as pd
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

PATH = "./cleaneddata"
df_jd = pd.read_csv(f"{PATH}/jobdesc.csv")
df_res = pd.read_csv(f"{PATH}/resume.csv")
df_know = pd.read_csv(f"{PATH}/knowledge.csv")
df_skills = pd.read_csv(f"{PATH}/skills.csv")
df_tech = pd.read_csv(f"{PATH}/techskills.csv")
df_alt = pd.read_excel(f"{PATH}/Alternate Titles.xlsx")

print(f"Loaded: {len(df_jd)} JDs, {len(df_res)} Resumes.")
print(f"doine O*NET Federal Database Online.")

Loaded: 2264 JDs, 2482 Resumes.
doine O*NET Federal Database Online.


In [3]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("Meowwww Mistral...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quant_config,
)
print("Model load.")

Meowwww Mistral...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model load.


In [35]:
alt_title_map = dict(zip(df_alt['Alternate Title'], df_alt['O*NET-SOC Code']))
soc_to_formal_title = dict(zip(df_alt['O*NET-SOC Code'], df_alt['Title']))

In [38]:
import pandas as pd
from thefuzz import process, fuzz
import re

# 1. FIX THE LOOKUPS (Key them by SOC Code, not Title)
tech_lookup = df_tech.groupby('O*NET-SOC Code')['Example'].apply(list).to_dict()
soft_skill_lookup = df_skills.groupby('O*NET-SOC Code')['Element Name'].apply(list).to_dict()

# Load the O*NET Rosetta Stone
df_alt = pd.read_excel(f"{PATH}/Alternate Titles.xlsx")
alt_title_map = dict(zip(df_alt['Alternate Title'], df_alt['O*NET-SOC Code']))
soc_to_formal_title = dict(zip(df_alt['O*NET-SOC Code'], df_alt['Title']))

def extract_resume_skills(resume_text):
    """The JSON Guillotine. Needs to be defined before inference."""
    prompt = f"""<s>[INST] Extract a flat JSON list of professional technical and soft skills from this resume.
    Resume: {resume_text[:2000]} [/INST] {{"candidate_skills": ["""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=400, temperature=0.1, pad_token_id=tokenizer.eos_token_id)
    res_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    try:
        match = re.search(r'\{.*\}', res_text, re.DOTALL)
        if match:
            return json.loads(match.group(0))['candidate_skills']
    except Exception:
        return []
    return []

def get_formal_soc_code(input_text):
    # 1. Mistral identifies the "vibe"
    prompt = f"<s>[INST] What is the most logical professional job title for this person? Output ONLY the title. \nText: {input_text[:1000]} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.1)
    vibe_title = tokenizer.decode(outputs[0], skip_special_tokens=True).split('[/INST]')[-1].strip()

    # 2. Fuzzy match vibe_title against the Alternate Titles database
    best_match, score = process.extractOne(vibe_title, alt_title_map.keys(), scorer=fuzz.token_set_ratio)

    soc_code = alt_title_map[best_match]
    formal_name = soc_to_formal_title[soc_code]

    return soc_code, formal_name, score

def run_live_inference(resume_str, jd_str=None):
    # 1. Get Candidate Skills
    can_skills = extract_resume_skills(resume_str)

    # 2. Identify Target Role
    anchor_text = jd_str if jd_str else resume_str
    soc_code, formal_title, confidence = get_formal_soc_code(anchor_text)

    # 3. Pull Ground Truth Requirements (NOW IT ACTUALLY MATCHES)
    req_tech = tech_lookup.get(soc_code, [])
    req_soft = soft_skill_lookup.get(soc_code, [])
    total_required = set(req_tech + req_soft)

    # 4. Calculate Gap
    current = set([str(s).lower().strip() for s in can_skills])
    gap = [str(s) for s in total_required if str(s).lower() not in current]

    return {
        "soc_code": soc_code,
        "formal_role": formal_title,
        "match_confidence": confidence,
        "candidate_skills_found": len(current),
        "gap": gap[:15], # Hard cap at 15 for the UI graph
        "trace": f"Grounded via O*NET SOC {soc_code} ({formal_title})"
    }